In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import twh_to_ej_str, build_fixed_output_xml, write_text, twh_to_ej, xy
from pathlib import Path

# Hydro

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

Hydro includes only conventional hydropower, excluding pumped-storage facilities. Assumptions for hydropower capacity and generation are based on historical data (ES) for 2020 and on projections from the 11th Basic Plan (BPESD) for the period 2025–2035.

In [7]:
dictGenGW = {2020: 1.8, 2025: 1.9, 2030: 1.9, 2035: 1.9}

In [8]:
dictGenTWh = {2020: 4.7, 2025: 3.7, 2030: 3.7, 2035: 3.7}

In [9]:
# Align years in case the dictionaries diverge later
years = sorted(set(dictGenGW) & set(dictGenTWh))
cap_gw = [dictGenGW[y] for y in years]
gen_twh = [dictGenTWh[y] for y in years]
cap_factor = [gen / (cap * 8.760) for cap, gen in zip(cap_gw, gen_twh)]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Capacity (GW)", "Generation (TWh)", "Implicit Capacity Factor")
)

fig.add_trace(go.Scatter(x=years, y=cap_gw, mode='lines+markers', name='Capacity'), row=1, col=1)
fig.add_trace(go.Scatter(x=years, y=gen_twh, mode='lines+markers', name='Generation'), row=1, col=2)
fig.add_trace(go.Scatter(x=years, y=cap_factor, mode='lines+markers', name='Capacity Factor'), row=1, col=3)

fig.update_yaxes(title_text='GW', row=1, col=1)
fig.update_yaxes(title_text='TWh', row=1, col=2)
fig.update_yaxes(title_text='Fraction', row=1, col=3)

fig.update_layout(
    template='plotly_white',
    width=1200, height=400,
    showlegend=False,
)

fig.show()


In [10]:
years = [2020, 2025, 2030, 2035]
dictFixedHydro = {y: twh_to_ej_str(dictGenTWh[y]) for y in years}
policy_name = "Nuclear-Ceiling"

xml_value = build_fixed_output_xml(
    dictFixedHydro,
    sector_name="electricity",
    subsector_name="hydro",
    tech_name="hydro",
)

xml_path = "../../input/policy/korea-2035/power/hydro_fixedOutput.xml"

write_text(xml_path, xml_value)

print("Wrote:", Path(xml_path).expanduser())

Wrote: ../../input/policy/korea-2035/power/hydro_fixedOutput.xml
